# V10-E5 — ActNN-style quantized activation storage (8-bit probe)

E1's verdict: exact dynamics, −62.1% VRAM, rejected on throughput (+48.9%).
The strategic implication: cut activation VRAM **without multiplying kernel
launches**. E5 stores linear/GELU activations as int8 (stochastic rounding,
unbiased) via custom autograd Functions — 2 cheap elementwise kernels per
tensor, zero new matmuls, zero recompute passes. Targets: mixer ~408 MB,
FFN ~277 MB (E0 map). Scan kernel interior untouched (V9 closed that).

**Decisions** (made per his "do it", open to correction): all-at-once scope
(all nn.Linear + nn.GELU) · per-channel scales · torch.rand stochastic
rounding (Triton fusion later) · 2-bit only if 8-bit passes E5.3.

**Gates:** E5.0 correctness+unbiasedness (fast fail) · E5.1 peak ≤ −25% vs
761 MB · E5.2 500-step lockstep max val dev ≤ 0.03 · E5.3 1500-step final
within 0.03 of fp32 ref and 1.319 · E5.4 2-bit probe (conditional).


In [ ]:
import os, urllib.request, time
import torch
import torch.nn as nn
import torch.nn.functional as F
assert torch.cuda.is_available(), "needs GPU"
device = "cuda"
def set_seed(s):
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(0)
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))


## 1. Data — campaign corpus A (vocab 104)

In [ ]:
def dl(url, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        print("downloading", url, flush=True)
        urllib.request.urlretrieve(url, path)
    return path
URLS = {
    "tinyshakespeare.txt": "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "alice.txt":           "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "frankenstein.txt":    "https://www.gutenberg.org/cache/epub/84/pg84.txt",
    "pride.txt":           "https://www.gutenberg.org/cache/epub/1342/pg1342.txt",
}
parts = []
for name, url in URLS.items():
    p = dl(url, f"data/{name}")
    with open(p, encoding="utf-8", errors="ignore") as f:
        parts.append(f.read())
text_a = "\n".join(parts)
chars_a = sorted(set(text_a)); vocab_a = len(chars_a)
stoi_a = {c: i for i, c in enumerate(chars_a)}
data_a = torch.tensor([stoi_a[c] for c in text_a], dtype=torch.long)
assert vocab_a == 104, f"vocab drift: {vocab_a}"
print(f"vocab {vocab_a} | tokens {len(data_a)/1e6:.1f}M")


## 2. Models + Triton kernels (byte-identical to E0/E1)

In [ ]:
"""
Linear-recurrent replacement for Transformer self-attention.

Experiment: take a standard decoder-only Transformer, rip out the
causal self-attention layer, and replace it with a *moving linear
equation* that compresses the past into a fixed-size running state:

    h_t = d * h_{t-1} + (1 - d) * (B x_t)        (linear recurrence)
    y_t = C h_t                                   (readout)

where d in (0,1)^S is a learned per-channel decay, B/C are learned
projections, and h has FIXED size S independent of sequence length.

This is O(T) time and O(1) memory w.r.t. context length, vs O(T^2)
for attention. The question: how much modeling power do we lose?
"""

import math
import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------------
# Token mixing layers
# ----------------------------------------------------------------------------

class SelectiveSegmentedState(nn.Module):
    """V5: gated segmented state-space mixer.

    Replaces the static decay with two dynamic mechanisms:

    1. Dynamic Selective Gate (the model's "will"):
         g_t = sigmoid(W_g x_t + b_g)      in (0,1), per token & channel
       acts as a learned per-token filter: low-information tokens get
       g_t -> 0 (forgotten fast), salient tokens get g_t -> 1 (kept).

    2. Hard Memory Reset Mask (segment boundaries):
         r_t = 1 if token t is a structural boundary (e.g. newline)
         h_t = (1 - r_t) * (g_t * h_{t-1} + (1 - g_t) * u_t)
       when r_t = 1 the state is multiplied by zero: the past memory
       buffer is completely destroyed and the new segment starts fresh.
       (The residual stream still carries x_t itself forward.)

    The recurrence stays affine in h (h_t = a_t*h_{t-1} + b_t with
    a_t = (1-r_t)*g_t, b_t = (1-r_t)*(1-g_t)*u_t), so the parallel
    associative scan still applies: O(T) work, log2(T) parallel passes.
    State dim stays frozen at `state_dim` (VRAM edge preserved).
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)  # bias = per-channel prior
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        # x: (B,T,dim); reset: (B,T) binary, 1 = boundary token
        u = self.in_proj(x)                    # (B,T,S)
        g = torch.sigmoid(self.gate_proj(x))   # (B,T,S) dynamic gate
        if reset is not None:
            keep = (1.0 - reset).unsqueeze(-1)  # (B,T,1)
            a = keep * g
            b = keep * (1.0 - g) * u
        else:
            a = g
            b = (1.0 - g) * u
        h_seq = parallel_scan_ab(a, b)          # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


class CausalSelfAttention(nn.Module):
    """Standard multi-head causal self-attention (the baseline)."""
    def __init__(self, dim, n_heads, dropout=0.0):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out = nn.Linear(dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                       # (B,T,H,D)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # (B,H,T,D)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)


def parallel_scan_ab(A, B):
    """Parallel associative scan (Hillis-Steele) for h_t = A_t*h_{t-1} + B_t.

    A, B: (B, T, S) per-step coefficients. Returns H: (B, T, S).
    log2(T) fully-parallel passes instead of T sequential steps.
    """
    Bsz, T, S = A.shape
    A = A.contiguous()
    Bh = B.contiguous()
    idx = torch.arange(T, device=A.device).view(1, T, 1)
    o = 1
    while o < T:
        m = idx >= o
        A_sh = torch.roll(A, o, dims=1)
        B_sh = torch.roll(Bh, o, dims=1)
        # compose segment [t-o, t-1] then [t]:
        #   a_new = a_t * a_{t-o},  b_new = a_t * b_{t-o} + b_t
        A_new = A * A_sh
        B_new = A * B_sh + Bh
        A = torch.where(m, A_new, A)
        Bh = torch.where(m, B_new, Bh)
        o *= 2
    return Bh


def parallel_linear_recurrence(u, decay):
    """Parallel associative scan (Hillis-Steele) for h_t = d*h_{t-1} + c_t.

    Reformulates the sequential loop as a tree reduction:
      - each step is an affine map  h -> a*h + b, packed as pair (a, b)
      - pair composition is associative:
            (a1,b1) then (a2,b2)  =  (a2*a1, a2*b1 + b2)
      - a parallel prefix over pairs yields every h_t at once

    log2(T) fully-parallel passes instead of T sequential steps.
    u: (B, T, S) inputs already projected; decay: (S,) in (0, 1).
    Returns h: (B, T, S) — identical math to the for-loop version.
    """
    Bsz, T, S = u.shape
    c = (1.0 - decay) * u
    A = decay.expand(Bsz, T, S).contiguous()   # a-coefficients
    return parallel_scan_ab(A, c)


class LinearRunningState(nn.Module):
    """Attention replacement: fixed-size running state, linear recurrence.

    h_t = decay * h_{t-1} + (1 - decay) * (B x_t)
    y_t = C h_t

    - decay: learned per-state-channel forget factor in (0, 1)
    - B:     input  projection  dim -> state_dim
    - C:     output projection  state_dim -> dim
    State size is FIXED (does not grow with context length).
    mode="loop": native Python for-loop (sequential, slow on GPU)
    mode="scan": parallel associative scan (log2(T) parallel passes)
    """
    def __init__(self, dim, state_dim=None, dropout=0.0, mode="loop"):
        super().__init__()
        self.mode = mode
        self.state_dim = state_dim or dim
        # init decay ~ U(0.9, 0.999) -> long-ish memory at start
        init = torch.empty(self.state_dim).uniform_(math.log(0.9 / 0.1),
                                                   math.log(0.999 / 0.001))
        self.logit_decay = nn.Parameter(init)
        self.in_proj = nn.Linear(dim, self.state_dim, bias=False)
        self.out_proj = nn.Linear(self.state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        B, T, _ = x.shape
        decay = torch.sigmoid(self.logit_decay)          # (S,) in (0,1)
        u = self.in_proj(x)                              # (B,T,S)
        if self.mode == "scan":
            h_seq = parallel_linear_recurrence(u, decay)  # (B,T,S)
        else:
            keep = 1.0 - decay
            h = torch.zeros(B, self.state_dim, device=x.device, dtype=x.dtype)
            outs = []
            for t in range(T):
                h = decay * h + keep * u[:, t]
                outs.append(h)
            h_seq = torch.stack(outs, dim=1)              # (B,T,S)
        return self.dropout(self.out_proj(h_seq))


# ----------------------------------------------------------------------------
# Blocks & models (identical except for the mixing layer)
# ----------------------------------------------------------------------------

class Block(nn.Module):
    def __init__(self, dim, n_heads, mixer, dropout=0.0, ffn_hidden=None):
        super().__init__()
        ffn_hidden = ffn_hidden or 4 * dim
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.mix = mixer
        self.mlp = nn.Sequential(
            nn.Linear(dim, ffn_hidden), nn.GELU(),
            nn.Linear(ffn_hidden, dim), nn.Dropout(dropout))

    def forward(self, x, reset=None):
        x = x + self.mix(self.ln1(x), reset)
        x = x + self.mlp(self.ln2(x))
        return x


class TinyLM(nn.Module):
    """Decoder-only LM. mixer_fn(dim, n_heads) builds the token-mixing layer."""
    def __init__(self, vocab, dim=128, n_layers=4, n_heads=4,
                 seq_len=128, mixer_fn=None, dropout=0.0, ffn_hidden=None,
                 newline_id=None):
        super().__init__()
        self.seq_len = seq_len
        self.newline_id = newline_id  # boundary token id for reset mask
        self.tok_emb = nn.Embedding(vocab, dim)
        self.pos_emb = nn.Embedding(seq_len, dim)
        self.blocks = nn.ModuleList(
            [Block(dim, n_heads, mixer_fn(dim, n_heads), dropout, ffn_hidden)
             for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(
            torch.arange(T, device=idx.device))[None]
        reset = None
        if self.newline_id is not None:
            reset = (idx == self.newline_id).float()  # (B,T) boundary mask
        for blk in self.blocks:
            x = blk(x, reset)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new, temp=1.0, top_k=None):
        for _ in range(max_new):
            ctx = idx[:, -self.seq_len:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / max(temp, 1e-6)
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def build_models(vocab, dim=128, n_layers=4, n_heads=4, seq_len=128,
                 state_dim=None, dropout=0.0):
    attn = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                  mixer_fn=lambda d, h: CausalSelfAttention(d, h, dropout),
                  dropout=dropout)
    lin = TinyLM(vocab, dim, n_layers, n_heads, seq_len,
                 mixer_fn=lambda d, h: LinearRunningState(d, state_dim, dropout),
                 dropout=dropout)
    return attn, lin


"""(inlined) V6/V7/V8-A: Triton-fused scans.

Fuses the V5 recurrence (per the V6 execution plan):
  Step A: the 256-wide hidden state lives in SRAM registers for the whole
          sequence -- loaded once per batch element, never spilled to HBM.
  Step B: sigmoid gate, hard reset mask, and the linear update
          h = keep*g*h + keep*(1-g)*u run in registers, every timestep.
  Step C: only h_t (plus a_t, saved for the backward pass) hit HBM.
          The parallel scan's O(log T) intermediate (B,T,S) tensors vanish.

Math is IDENTICAL to V5's SelectiveSegmentedState -- only the execution
path changes. Zero new parameters.
"""
import triton
import triton.language as tl

BLOCK_S = 256  # state dim; the whole state fits in SRAM


@triton.jit
def _sel_scan_fwd_kernel(u_ptr, gp_ptr, keep_ptr, h_ptr, a_ptr,
                         T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with a_t = keep_t*g_t,
    b_t = keep_t*(1-g_t)*u_t, g_t = sigmoid(gp_t)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # Step A: state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))              # Step B: gate in registers
        at = k * g
        h = at * h + k * (1.0 - g) * u_t             # Step B: update in registers
        tl.store(h_b + t * BLOCK + offs, h)          # Step C: outputs to HBM
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _sel_scan_bwd_kernel(dh_ptr, hs_ptr, ap_ptr, u_ptr, gp_ptr, keep_ptr,
                         du_ptr, dgp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    ap is (B,T+1,S) with ap[t+1] = a_{t+1} (ap[T] = 0).
    D_t = gout_t + a_{t+1}*D_{t+1}; du_t = k(1-g)D_t;
    dgp_t = k*g*(1-g)*(h_{t-1}-u_t)*D_t."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    ap_b = ap_ptr + b * bsp
    u_b = u_ptr + b * bs
    gp_b = gp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dgp_b = dgp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(ap_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        gp_t = tl.load(gp_b + t * BLOCK + offs).to(tl.float32)
        k = tl.load(k_b + t).to(tl.float32)
        g = 1.0 / (1.0 + tl.exp(-gp_t))
        omg = 1.0 - g
        tl.store(du_b + t * BLOCK + offs, k * omg * D)
        tl.store(dgp_b + t * BLOCK + offs, k * g * omg * (h_prev - u_t) * D)
        dnext = D


class _SelScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, gp, keep):
        assert u.is_cuda and u.dtype == torch.float32, "V6 kernel: CUDA fp32"
        assert u.shape == gp.shape and u.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, gp, keep = u.contiguous(), gp.contiguous(), keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _sel_scan_fwd_kernel[(B,)](u, gp, keep, h, a, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, gp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, gp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h    # hs[t] = h_{t-1}
        ap = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        ap[:, :T] = a    # ap[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dgp = torch.empty_like(gp)
        _sel_scan_bwd_kernel[(B,)](dh, hs, ap, u, gp, keep, du, dgp, T,
                                   BLOCK=BLOCK_S, num_warps=4)
        return du, dgp, None


def triton_selective_scan(u, gp, keep):
    """Fused selective scan. u, gp: (B,T,256) fp32 CUDA; keep: (B,T) fp32.
    Returns h: (B,T,256). Differentiable."""
    return _SelScanFn.apply(u, gp, keep)


class SelectiveSegmentedStateTriton(nn.Module):
    """V6 mixer: IDENTICAL math and parameters to V5's SelectiveSegmentedState.

    The only change is execution: the recurrence runs in the fused Triton
    kernel (Step A/B/C) instead of the PyTorch Hillis-Steele scan.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V6 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = self.gate_proj(x).float().contiguous()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)
        return self.dropout(self.out_proj(h_seq.to(x.dtype)))


class SelectiveSegmentedStateV7(nn.Module):
    """V7 mixer: V6's Triton-fused selective scan + two literature grafts.

    Graft #1 — output gate + state normalization (HGRN2 / Mamba-2 / GDN):
        y_t = C · (o_t ⊙ RMSNorm(h_t)),   o_t = σ(W_o x_t + b_o)
    States accumulate over segments so readout scale drifts; every strong
    post-Mamba block gates the readout. RMSNorm here is parameter-free
    (no scale vector) to keep the parameter budget exact.

    Graft #3 — layer-graded forget floor (HGRN, NeurIPS'23):
        g_t = σ(W_g x_t + b_g + floor_l)
    floor_l is a per-layer learnable scalar, initialized monotonically
    increasing bottom→top, so low layers forget fast (spelling-level) and
    top layers retain (phrase-level). Complements the newline hard reset.

    The recurrence math is UNCHANGED → the same fused Triton kernel as V6.
    With FFN 1023 the budget lands EXACTLY on the attention baseline.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V7 kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)
        self.gate_proj = nn.Linear(dim, state_dim)
        self.out_gate_proj = nn.Linear(dim, state_dim)
        self.out_proj = nn.Linear(state_dim, dim, bias=False)
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        gp = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_selective_scan(u, gp, keep)          # (B,T,S)
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


# ----------------------------------------------------------------------------
# V8-A: diagonal gated delta-rule scan
# ----------------------------------------------------------------------------

@triton.jit
def _delta_scan_fwd_kernel(u_ptr, ap_ptr, bp_ptr, kp_ptr, keep_ptr,
                           h_ptr, a_ptr, T, BLOCK: tl.constexpr):
    """Forward: h_t = a_t*h_{t-1} + b_t with
        alpha = sigmoid(ap_t), beta = sigmoid(bp_t), kk = sigmoid(kp_t),
        v = silu(u_t),
        a_t = keep_t * alpha * (1 - beta*kk^2),
        b_t = keep_t * beta * kk * v.
    Diagonal restriction of Gated DeltaNet's S_t = a_t S_{t-1}(I-b_t k k^T)
    + b_t v k^T to per-channel 1x1 systems. Strictly contractive:
    sigmoid bounds keep (1-beta*kk^2) in (0,1)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    h_b = h_ptr + b * bs
    a_b = a_ptr + b * bs
    k_b = keep_ptr + b * T
    h = tl.zeros([BLOCK], dtype=tl.float32)          # state in SRAM
    for t in range(T):
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su                                 # SiLU in registers
        at = kr * alpha * (1.0 - beta * kk * kk)
        bt = kr * beta * kk * v
        h = at * h + bt
        tl.store(h_b + t * BLOCK + offs, h)
        tl.store(a_b + t * BLOCK + offs, at)


@triton.jit
def _delta_scan_bwd_kernel(dh_ptr, hs_ptr, apn_ptr, u_ptr, ap_ptr, bp_ptr,
                           kp_ptr, keep_ptr, du_ptr, dap_ptr, dbp_ptr,
                           dkp_ptr, T, BLOCK: tl.constexpr):
    """Backward. hs is (B,T+1,S) with hs[t] = h_{t-1} (hs[0] = 0);
    apn is (B,T+1,S) with apn[t+1] = a_{t+1} (apn[T] = 0).
    Adjoint: D_t = gout_t + a_{t+1}*D_{t+1}; then per-step
      du  = D*kr*beta*kk*silu'(u)
      dap = D*h_prev*kr*alpha*(1-alpha)*(1-beta*kk^2)
      dbp = D*kr*beta*(1-beta)*(kk*v - alpha*kk^2*h_prev)
      dkp = D*kr*beta*kk*(1-kk)*(v - 2*alpha*kk*h_prev)
    (hand-derived; verified against autograd on CPU before transcription)."""
    b = tl.program_id(0)
    offs = tl.arange(0, BLOCK)
    bs = T * BLOCK
    bsp = (T + 1) * BLOCK
    dh_b = dh_ptr + b * bs
    hs_b = hs_ptr + b * bsp
    apn_b = apn_ptr + b * bsp
    u_b = u_ptr + b * bs
    ap_b = ap_ptr + b * bs
    bp_b = bp_ptr + b * bs
    kp_b = kp_ptr + b * bs
    k_b = keep_ptr + b * T
    du_b = du_ptr + b * bs
    dap_b = dap_ptr + b * bs
    dbp_b = dbp_ptr + b * bs
    dkp_b = dkp_ptr + b * bs
    dnext = tl.zeros([BLOCK], dtype=tl.float32)      # D_{t+1}, SRAM-resident
    for ti in range(T):
        t = T - 1 - ti
        gout = tl.load(dh_b + t * BLOCK + offs).to(tl.float32)
        a_next = tl.load(apn_b + (t + 1) * BLOCK + offs).to(tl.float32)
        D = gout + a_next * dnext
        h_prev = tl.load(hs_b + t * BLOCK + offs).to(tl.float32)
        u_t = tl.load(u_b + t * BLOCK + offs).to(tl.float32)
        ap_t = tl.load(ap_b + t * BLOCK + offs).to(tl.float32)
        bp_t = tl.load(bp_b + t * BLOCK + offs).to(tl.float32)
        kp_t = tl.load(kp_b + t * BLOCK + offs).to(tl.float32)
        kr = tl.load(k_b + t).to(tl.float32)
        alpha = 1.0 / (1.0 + tl.exp(-ap_t))
        beta = 1.0 / (1.0 + tl.exp(-bp_t))
        kk = 1.0 / (1.0 + tl.exp(-kp_t))
        su = 1.0 / (1.0 + tl.exp(-u_t))
        v = u_t * su
        silu_p = su * (1.0 + u_t * (1.0 - su))
        k2 = kk * kk
        omb = 1.0 - beta * k2
        tl.store(du_b + t * BLOCK + offs,
                 D * kr * beta * kk * silu_p)
        tl.store(dap_b + t * BLOCK + offs,
                 D * h_prev * kr * alpha * (1.0 - alpha) * omb)
        tl.store(dbp_b + t * BLOCK + offs,
                 D * kr * beta * (1.0 - beta) * (kk * v - alpha * k2 * h_prev))
        tl.store(dkp_b + t * BLOCK + offs,
                 D * kr * beta * kk * (1.0 - kk) * (v - 2.0 * alpha * kk * h_prev))
        dnext = D


def _delta_scan_cpu(u, ap, bp, kp, keep):
    """CPU reference for the delta scan: differentiable pure-PyTorch loop.

    Used ONLY for local (CPU) verification. On CUDA the fused Triton
    kernel above is used instead (gate-checked against this math on Colab).
    """
    B, T, S = u.shape
    kr = keep.unsqueeze(-1)
    alpha = torch.sigmoid(ap)
    beta = torch.sigmoid(bp)
    kk = torch.sigmoid(kp)
    v = u * torch.sigmoid(u)                          # SiLU
    h = torch.zeros(B, S, dtype=u.dtype, device=u.device)
    hs = []
    for t in range(T):
        a = kr[:, t] * alpha[:, t] * (1.0 - beta[:, t] * kk[:, t] ** 2)
        bb = kr[:, t] * beta[:, t] * kk[:, t] * v[:, t]
        h = a * h + bb
        hs.append(h)
    return torch.stack(hs, dim=1)


class _DeltaScanFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, u, ap, bp, kp, keep):
        assert u.is_cuda and u.dtype == torch.float32, \
            "V8-A kernel: CUDA fp32"
        for t in (u, ap, bp, kp):
            assert t.shape == u.shape and t.dim() == 3
        assert keep.shape == u.shape[:2]
        assert u.shape[2] == BLOCK_S
        B, T, S = u.shape
        u, ap, bp, kp = u.contiguous(), ap.contiguous(), bp.contiguous(), kp.contiguous()
        keep = keep.contiguous()
        h = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        a = torch.empty(B, T, S, device=u.device, dtype=torch.float32)
        _delta_scan_fwd_kernel[(B,)](u, ap, bp, kp, keep, h, a, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        ctx.save_for_backward(u, ap, bp, kp, keep, h, a)
        return h

    @staticmethod
    def backward(ctx, dh):
        u, ap, bp, kp, keep, h, a = ctx.saved_tensors
        B, T, S = u.shape
        dh = dh.contiguous()
        hs = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        hs[:, 1:] = h      # hs[t] = h_{t-1}
        apn = torch.zeros(B, T + 1, S, device=u.device, dtype=torch.float32)
        apn[:, :T] = a     # apn[t+1] = a_{t+1}
        du = torch.empty_like(u)
        dap = torch.empty_like(ap)
        dbp = torch.empty_like(bp)
        dkp = torch.empty_like(kp)
        _delta_scan_bwd_kernel[(B,)](dh, hs, apn, u, ap, bp, kp, keep,
                                    du, dap, dbp, dkp, T,
                                    BLOCK=BLOCK_S, num_warps=4)
        return du, dap, dbp, dkp, None


def triton_delta_scan(u, ap, bp, kp, keep):
    """Fused diagonal delta-rule scan.

    u: raw Bx (SiLU applied in-kernel); ap/bp/kp: pre-sigmoid
    alpha/beta/key projections; keep: (B,T) reset mask.
    (B,T,256) fp32; CUDA -> Triton kernel, CPU -> differentiable
    reference loop (verification only). Returns h: (B,T,256).
    """
    if u.is_cuda:
        return _DeltaScanFn.apply(u, ap, bp, kp, keep)
    return _delta_scan_cpu(u, ap, bp, kp, keep)


class SelectiveSegmentedStateV8A(nn.Module):
    """V8-A mixer: diagonal gated delta rule on V7's chassis.

    Per channel (delta rule = erase-before-write, Gated DeltaNet Eq.10
    restricted to 1x1 systems):
        alpha_t = sigmoid(W_a x_t + b_a + floor_l)   # reuse gate_proj+floor
        beta_t  = sigmoid(W_b x_t + b_b),  b_b = 0   # NEW write_proj
        k_t     = sigmoid(W_k x_t + b_k),  b_k = +2  # NEW key_proj
        v_t     = SiLU(B x_t)                        # reuse in_proj + SiLU
        a_t = alpha_t * (1 - beta_t * k_t^2)
        b_t = beta_t * k_t * v_t
        h_t = (1 - r_t) * (a_t .* h_{t-1} + b_t)
        y_t = C . (o_t .* RMSNorm(h_t)), o_t = sigmoid(W_o x_t)

    Init puts the model near V7's convex-blend regime (k~=0.88, beta~=0.5)
    and the sigmoid bounds keep every step strictly contractive.
    With FFN 766 the budget lands within the relaxed <=3000 gate.
    """
    def __init__(self, dim, state_dim=256, dropout=0.0, forget_floor_init=0.0):
        super().__init__()
        assert state_dim == BLOCK_S, "V8-A kernel fuses a 256-wide state"
        self.state_dim = state_dim
        self.in_proj = nn.Linear(dim, state_dim, bias=False)   # B
        self.gate_proj = nn.Linear(dim, state_dim)             # W_alpha
        self.write_proj = nn.Linear(dim, state_dim)            # W_beta (NEW)
        self.key_proj = nn.Linear(dim, state_dim)              # W_k (NEW)
        self.out_gate_proj = nn.Linear(dim, state_dim)         # W_o
        self.out_proj = nn.Linear(state_dim, dim, bias=False)  # C
        self.forget_floor = nn.Parameter(
            torch.tensor(float(forget_floor_init)))
        self.write_proj.bias.data.zero_()       # b_beta = 0  -> beta ~= 0.5
        self.key_proj.bias.data.fill_(2.0)       # b_k = +2.0  -> k ~= 0.88
        self.dropout = nn.Dropout(dropout)
        self.eps = 1e-6
        self.last_state = None  # stashed h_seq for stability logging (no grad)

    def forward(self, x, reset=None):
        u = self.in_proj(x).float().contiguous()
        ap = (self.gate_proj(x) + self.forget_floor).float().contiguous()
        bp = self.write_proj(x).float().contiguous()
        kp = self.key_proj(x).float().contiguous()
        og = self.out_gate_proj(x).float()
        if reset is not None:
            keep = (1.0 - reset).float().contiguous()
        else:
            keep = torch.ones(x.shape[0], x.shape[1],
                              device=x.device, dtype=torch.float32)
        h_seq = triton_delta_scan(u, ap, bp, kp, keep)      # (B,T,S)
        self.last_state = h_seq.detach()  # stability logging only; no numerics change
        h_norm = h_seq / torch.sqrt(
            (h_seq ** 2).mean(dim=-1, keepdim=True) + self.eps)
        y = self.out_proj((torch.sigmoid(og) * h_norm).to(x.dtype))
        return self.dropout(y)


## 3. qstore — quantized autograd wrappers (CPU-verified pre-flight)

In [ ]:
"""V10-E5: ActNN-style quantized activation storage (stochastic rounding).

Wraps nn.Linear / nn.GELU so autograd saves quantized + scales instead of
fp32 activations. Dynamics are UNBIASED via stochastic rounding
(E[floor(x+U)] = x for U ~ Uniform[0,1)). No extra matmuls, no recompute
passes: 2 cheap elementwise kernels per tensor (quantize fwd, dequant bwd).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

# active bit-width; set via set_qbits()
_QBITS = 8


def set_qbits(b):
    assert b in (8, 2), b
    global _QBITS
    _QBITS = b


def quantize_stochastic(x, bits=8):
    """Quantize x (per-channel, stochastic rounding).

    8-bit: returns (int8 tensor, scale fp32, None).
    2-bit: returns (packed uint8, scale fp32, orig_last_dim) — 4 vals/byte.
    Unbiased: E[dequant(quantize(x))] = x (no clamping bias: xs never exceeds
    [qmin, qmax] because scale = amax/qmax).
    """
    assert bits in (8, 2), bits
    qmax = 2 ** (bits - 1) - 1
    qmin = -(2 ** (bits - 1))
    amax = x.abs().amax(dim=-1, keepdim=True).clamp_min(1e-8)
    scale = amax / qmax
    xs = x / scale
    x_q = torch.floor(xs + torch.rand_like(xs)).clamp(qmin, qmax)
    if bits == 8:
        return x_q.to(torch.int8), scale, None
    # 2-bit: map {-2,-1,0,1} -> {0,1,2,3}, pack 4 per byte along last dim
    x_u = (x_q + 2).to(torch.uint8)
    C = x_u.size(-1)
    pad = (-C) % 4
    if pad:
        x_u = F.pad(x_u, (0, pad))
    x_u = x_u.reshape(*x_u.shape[:-1], -1, 4)
    packed = (x_u[..., 0] | (x_u[..., 1] << 2)
              | (x_u[..., 2] << 4) | (x_u[..., 3] << 6))
    return packed, scale, C


def dequantize(data, scale, bits, meta):
    if bits == 8:
        return data.to(torch.float32) * scale
    C = meta
    b = data.to(torch.uint8)
    v = torch.stack([(b & 0b11).to(torch.int8) - 2,
                     ((b >> 2) & 0b11).to(torch.int8) - 2,
                     ((b >> 4) & 0b11).to(torch.int8) - 2,
                     ((b >> 6) & 0b11).to(torch.int8) - 2], dim=-1)
    x_q = v.reshape(*v.shape[:-2], -1)[..., :C]
    return x_q.to(torch.float32) * scale


class QLinearFn(torch.autograd.Function):
    """F.linear with quantized-stored input activation (weight kept fp32)."""

    @staticmethod
    def forward(ctx, x, weight, bias):
        bits = _QBITS
        data, scale, meta = quantize_stochastic(x, bits)
        ctx.bits, ctx.meta = bits, meta
        ctx.has_bias = bias is not None
        if bias is not None:
            ctx.save_for_backward(data, scale, weight, bias)
        else:
            ctx.save_for_backward(data, scale, weight)
        return F.linear(x, weight, bias)

    @staticmethod
    def backward(ctx, grad_out):
        saved = ctx.saved_tensors
        data, scale, weight = saved[0], saved[1], saved[2]
        x = dequantize(data, scale, ctx.bits, ctx.meta)
        grad_x = grad_out @ weight
        go = grad_out.reshape(-1, grad_out.size(-1))
        xr = x.reshape(-1, x.size(-1))
        grad_w = go.t() @ xr
        grad_b = None
        if ctx.has_bias:
            grad_b = grad_out.sum(dim=tuple(range(grad_out.dim() - 1)))
        return grad_x, grad_w, grad_b


class QLinear(nn.Module):
    """Drop-in nn.Linear replacement sharing the original Parameters."""

    def __init__(self, lin: nn.Linear):
        super().__init__()
        self.weight = lin.weight
        self.bias = lin.bias
        self.in_features = lin.in_features
        self.out_features = lin.out_features

    def forward(self, x):
        return QLinearFn.apply(x, self.weight, self.bias)


class QGELUFn(torch.autograd.Function):
    """nn.GELU(approximate='none') with quantized-stored input activation."""

    @staticmethod
    def forward(ctx, x):
        bits = _QBITS
        data, scale, meta = quantize_stochastic(x, bits)
        ctx.bits, ctx.meta = bits, meta
        ctx.save_for_backward(data, scale)
        return F.gelu(x)

    @staticmethod
    def backward(ctx, grad_out):
        data, scale = ctx.saved_tensors
        x = dequantize(data, scale, ctx.bits, ctx.meta)
        return torch.ops.aten.gelu_backward(grad_out, x, approximate="none")


class QGELU(nn.Module):
    def forward(self, x):
        return QGELUFn.apply(x)


def quantize_model(model):
    """Recursively replace nn.Linear -> QLinear, nn.GELU -> QGELU in-place.

    Returns (n_linear, n_gelu) replaced. Parameters are shared, not copied.
    """
    n_lin = n_gelu = 0
    for name, child in list(model.named_children()):
        if isinstance(child, nn.Linear):
            setattr(model, name, QLinear(child))
            n_lin += 1
        elif isinstance(child, nn.GELU):
            setattr(model, name, QGELU())
            n_gelu += 1
        else:
            a, b = quantize_model(child)
            n_lin += a
            n_gelu += b
    return n_lin, n_gelu


## 4. Builders + lockstep data

`build_v8a(seed, quantized)`: identical init both arms; quantization applied
after build, optimizer constructed after. 1500 pre-generated batch-32s, one
seeded order, shared by both arms. Fixed 20-batch val set.


In [ ]:
DIM, LAYERS, HEADS, SEQ, FFN_H = 256, 8, 8, 128, 766
N_STEPS = 1500

def build_v8a(seed, quantized=False, bits=8):
    set_seed(seed)
    m = TinyLM(vocab_a, DIM, LAYERS, HEADS, SEQ,
               mixer_fn=lambda d, h: SelectiveSegmentedStateV8A(d, state_dim=256),
               ffn_hidden=FFN_H, newline_id=stoi_a.get("\n")).to(device)
    import math as _math
    _floors = torch.linspace(_math.log(0.3/0.7), _math.log(0.9/0.1), LAYERS)
    for _i, _blk in enumerate(m.blocks):
        _blk.mix.forget_floor.data.fill_(_floors[_i])
    if quantized:
        set_qbits(bits)
        nl, ng = quantize_model(m)
        print(f"quantized: {nl} linears + {ng} gelus -> int{bits}")
    o = torch.optim.AdamW(m.parameters(), lr=3e-4)
    return m, o

def pregen(d, seq_len, bs, n, seed, split="train"):
    dd = d[:int(0.95*len(d))] if split == "train" else d[int(0.95*len(d)):]
    g = torch.Generator().manual_seed(seed)
    out = []
    for _ in range(n):
        i = torch.randint(0, len(dd)-seq_len-1, (bs,), generator=g)
        out.append((torch.stack([dd[j:j+seq_len] for j in i]),
                    torch.stack([dd[j+1:j+seq_len+1] for j in i])))
    return out

train_batches = pregen(data_a, SEQ, 32, N_STEPS, 1234, "train")
val_batches = pregen(data_a, SEQ, 32, 20, 999, "val")
print(f"batches: train {len(train_batches)}, val {len(val_batches)}")

def val_loss(m):
    tot = 0.0
    with torch.no_grad():
        for xb, yb in val_batches:
            _, loss = m(xb.to(device), yb.to(device))
            tot += float(loss)
    return tot / len(val_batches)

def train_step(m, o, xb, yb):
    o.zero_grad(set_to_none=True)
    _, loss = m(xb, yb)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
    o.step()
    return float(loss)


## E5.0 — Quantizer correctness on REAL activations (fast fail)

Roundtrip stats on captured linear inputs (mixer in_proj, FFN up, head) +
GELU input: max/mean abs err, mean signed err (≈ 0 = unbiased), clamp-hit %.
Separate-model one-step grad comparison vs fp32 (expect small, unbiased).


In [ ]:
set_qbits(8)
m_probe, _ = build_v8a(0, quantized=False)
captured = {}
hooks = []
def _hook(name):
    def fn(mod, inp, out):
        captured[name] = inp[0].detach()
    return fn
targets = [("mixer_in", m_probe.blocks[0].mix.in_proj),
           ("ffn_up", m_probe.blocks[0].mlp[0]),
           ("head", m_probe.head)]
for name, mod in targets:
    hooks.append(mod.register_forward_hook(_hook(name)))
xb0, yb0 = train_batches[0][0].to(device), train_batches[0][1].to(device)
with torch.no_grad():
    m_probe(xb0)
for h in hooks:
    h.remove()
n_params_before = sum(p.numel() for p in m_probe.parameters())

ok = True
for name, act in captured.items():
    d, s, mt = quantize_stochastic(act, 8)
    r = dequantize(d, s, 8, mt)
    err = (r - act)
    mx, mn, ms = err.abs().max().item(), err.abs().mean().item(), err.mean().item()
    sc = act.abs().amax(dim=-1, keepdim=True).clamp_min(1e-8) / 127
    rel = (err.abs() / (act.abs() + sc)).max().item()
    print(f"{name:10s} shape {tuple(act.shape)}  max|e|={mx:.3e} mean|e|={mn:.3e} "
          f"signed={ms:.2e} maxrel={rel:.3f}")
    if abs(ms) > 5e-4:
        ok = False; print("  FAIL: biased!")
print("E5.0 unbiasedness:", "PASS" if ok else "FAIL")

# grad closeness: separate identical-init models, one step, same batch
mf, of = build_v8a(0, quantized=False)
mq, oq = build_v8a(0, quantized=True, bits=8)
with torch.no_grad():
    mf(xb0); mq(xb0)  # warmup (Triton compile)
mf, of = build_v8a(0, quantized=False)
mq, oq = build_v8a(0, quantized=True, bits=8)
train_step(mf, of, xb0, yb0)
train_step(mq, oq, xb0, yb0)
num = den = 0.0
for pf, pq in zip(mf.parameters(), mq.parameters()):
    num += (pf.grad - pq.grad).pow(2).sum().item()
    den += pf.grad.pow(2).sum().item()
rel_grad = (num / (den + 1e-12)) ** 0.5
print(f"E5.0 one-step grad rel-delta vs fp32: {rel_grad:.3e} (info; expect small, nonzero)")


## E5.1 — VRAM probe (gate: peak ≤ −25% vs 761 MB baseline)

In [ ]:
mq, oq = build_v8a(0, quantized=True, bits=8)
xb0, yb0 = train_batches[0][0].to(device), train_batches[0][1].to(device)
with torch.no_grad():
    mq(xb0)  # warmup outside measurement
torch.cuda.reset_peak_memory_stats()
for s in range(3):
    xb, yb = train_batches[s][0].to(device), train_batches[s][1].to(device)
    train_step(mq, oq, xb, yb)
peak_q = torch.cuda.max_memory_allocated() / 1e6
reduction = 100 * (761 - peak_q) / 761
print(f"E5.1 peak quantized: {peak_q:.0f} MB vs 761 MB baseline")
print(f"E5.1 reduction: {reduction:.1f}%  (gate >= 25%)")
e51_pass = reduction >= 25


## E5.2/E5.3 — 1500-step lockstep: quantized vs fp32

Same 1500 batches, same order, val every 100. E5.2 gate at step 500
(max val dev ≤ 0.03); E5.3 gate at 1500 (final within 0.03 of fp32 ref
and within 0.05 of the 1.319 V8-A anchor — looser externally since data
order differs from the original run).


In [ ]:
results = {}
for tag, quant in (("fp32", False), ("q8", True)):
    m, o = build_v8a(0, quantized=quant, bits=8)
    xb0, yb0 = train_batches[0][0].to(device), train_batches[0][1].to(device)
    with torch.no_grad():
        m(xb0)  # warmup outside timing
    vals = [val_loss(m)]
    t0 = time.perf_counter()
    for s in range(N_STEPS):
        xb, yb = train_batches[s][0].to(device), train_batches[s][1].to(device)
        train_step(m, o, xb, yb)
        if (s + 1) % 100 == 0:
            vals.append(val_loss(m))
            print(f"[{tag}] step {s+1:4d} val {vals[-1]:.4f}", flush=True)
    dt = time.perf_counter() - t0
    results[tag] = {"vals": vals, "secs": dt}
    print(f"[{tag}] done {dt:.0f}s ({1000*dt/N_STEPS:.1f} ms/step)", flush=True)

vf, vq = results["fp32"]["vals"], results["q8"]["vals"]
dev500 = max(abs(a - b) for a, b in zip(vq[:6], vf[:6]))
dev1500 = abs(vq[-1] - vf[-1])
anchor = abs(vq[-1] - 1.319)
overhead = 100 * (results["q8"]["secs"] - results["fp32"]["secs"]) / results["fp32"]["secs"]
print(f"E5.2 max val dev @500: {dev500:.4f} (gate <= 0.03)")
print(f"E5.3 final q8 {vq[-1]:.4f} vs fp32 {vf[-1]:.4f}  dev {dev1500:.4f} (gate <= 0.03)")
print(f"E5.3 vs 1.319 anchor: {anchor:.4f} (gate <= 0.05)")
print(f"E5 cost: step overhead {overhead:+.1f}% (expect +3..8%; falsify > 15%)")
e52_pass = dev500 <= 0.03
e53_pass = e52_pass and dev1500 <= 0.03 and anchor <= 0.05 and overhead <= 15


## E5.4 — 2-bit probe (conditional on E5.3)

Correctness on real activations + 500-step lockstep probe at 2-bit.


In [ ]:
e54 = {}
if e53_pass:
    set_qbits(2)
    m2, o2 = build_v8a(0, quantized=True, bits=2)
    xb0, yb0 = train_batches[0][0].to(device), train_batches[0][1].to(device)
    with torch.no_grad():
        m2(xb0)
    torch.cuda.reset_peak_memory_stats()
    for s in range(3):
        xb, yb = train_batches[s][0].to(device), train_batches[s][1].to(device)
        train_step(m2, o2, xb, yb)
    peak2 = torch.cuda.max_memory_allocated() / 1e6
    vals2, valsf = [], []
    m2, o2 = build_v8a(0, quantized=True, bits=2)
    mf2, of2 = build_v8a(0, quantized=False)
    for mm, oo in ((m2, o2), (mf2, of2)):
        with torch.no_grad():
            mm(xb0)
    for s in range(500):
        xb, yb = train_batches[s][0].to(device), train_batches[s][1].to(device)
        train_step(m2, o2, xb, yb)
        train_step(mf2, of2, xb, yb)
        if (s + 1) % 100 == 0:
            vals2.append(val_loss(m2)); valsf.append(val_loss(mf2))
            print(f"[2bit] step {s+1:4d} val {vals2[-1]:.4f} vs fp32 {valsf[-1]:.4f}", flush=True)
    dev2 = max(abs(a - b) for a, b in zip(vals2, valsf))
    e54 = {"peak2": peak2, "dev2": dev2,
           "pass": dev2 <= 0.05 and peak2 <= 0.5 * 761}
    print(f"E5.4 2-bit: peak {peak2:.0f} MB, max val dev {dev2:.4f} -> "
          f"{'PASS' if e54['pass'] else 'FAIL'}")
else:
    print("E5.4 SKIPPED (E5.3 did not pass)")


## E5.5 — Verdict + save

In [ ]:
print(f"GATE E5.0 unbiasedness        : {'PASS' if ok else 'FAIL'}")
print(f"GATE E5.1 peak reduction >=25% : {'PASS' if e51_pass else 'FAIL'} ({reduction:.1f}%)")
print(f"GATE E5.2 500-step dev <=0.03  : {'PASS' if e52_pass else 'FAIL'} ({dev500:.4f})")
print(f"GATE E5.3 1500-step re-gate    : {'PASS' if e53_pass else 'FAIL'}")
if e54:
    print(f"GATE E5.4 2-bit probe          : {'PASS' if e54['pass'] else 'FAIL'}")
e5_pass = ok and e51_pass and e52_pass and e53_pass
print("E5 VERDICT:", "PASS — quantized activation storage is a production lever"
      if e5_pass else "FAIL — see gate(s) above")
lines = [f"e50_unbiased={ok}", f"e50_rel_grad={rel_grad:.3e}",
         f"e51_peak_MB={peak_q:.0f}", f"e51_reduction_pct={reduction:.1f}",
         f"e52_dev500={dev500:.4f}", f"e53_dev1500={dev1500:.4f}",
         f"e53_anchor={anchor:.4f}", f"e53_overhead_pct={overhead:.1f}",
         f"e53_final_q8={vq[-1]:.4f}", f"e53_final_fp32={vf[-1]:.4f}"]
if e54:
    lines += [f"e54_peak2_MB={e54['peak2']:.0f}", f"e54_dev2={e54['dev2']:.4f}",
              f"e54_pass={e54['pass']}"]
lines.append(f"gates_pass={bool(e5_pass)}")
open("/content/e5_results.txt", "w").write("\n".join(lines))
print(open("/content/e5_results.txt").read())
from google.colab import files
files.download("/content/e5_results.txt")
print("SAVE DONE")
